<a href="https://colab.research.google.com/github/DigoShane/GitHub-ML/blob/main/2d_nn_withdomaindiscretization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# CELL 1 — INSTALL AND IMPORT PACKAGES
# ============================================================

!pip -q install --upgrade plotly ipywidgets

from google.colab import output
output.enable_custom_widget_manager()

import numpy as np
import plotly
import plotly.graph_objects as go
import ipywidgets as widgets

from IPython.display import display, clear_output

print("Plotly version:", plotly.__version__)
print("Widget manager enabled.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 61.3 MB/s eta 0:00:00
Plotly version: 6.9.0
Widget manager enabled.


In [2]:
# ============================================================
# CELL 2 — USER SETTINGS
# ============================================================

# ------------------------------------------------------------
# Neural-network architecture
# ------------------------------------------------------------

# Number of nodes in hidden layer 1
N1 = 2

# Number of nodes in hidden layer 2
N2 = 2


# ------------------------------------------------------------
# Activation functions
#
# Options:
#     "tanh"
#     "sigmoid"
#     "softplus"
#     "relu"
#     "silu"
# ------------------------------------------------------------

ACTIVATION_LAYER_1 = "tanh"
ACTIVATION_LAYER_2 = "tanh"


# ------------------------------------------------------------
# Softplus parameters
#
# These values are used only when the corresponding
# activation is set to "softplus".
# ------------------------------------------------------------

SOFTPLUS_BETA_1 = 1.0
SOFTPLUS_BETA_2 = 1.0


# ------------------------------------------------------------
# Weight and bias slider ranges
#
# Weight sliders:
#     -WEIGHT_MAX <= weight <= WEIGHT_MAX
#
# Bias sliders:
#     -BIAS_MAX <= bias <= BIAS_MAX
# ------------------------------------------------------------

WEIGHT_MAX = 5.0
BIAS_MAX = 5.0

# Increment used by the sliders
SLIDER_STEP = 0.1


# ------------------------------------------------------------
# Random initialization
#
# The initial values are randomly selected from:
#
# [-INITIAL_PARAMETER_SCALE, INITIAL_PARAMETER_SCALE]
# ------------------------------------------------------------

INITIAL_PARAMETER_SCALE = 1.0
RANDOM_SEED = 42


# ------------------------------------------------------------
# Domain and plotting settings
# ------------------------------------------------------------

# Number of plotting points in each coordinate direction
GRID_POINTS = 50

# Plot dimensions
PLOT_WIDTH = 1100
PLOT_HEIGHT = 850


# ------------------------------------------------------------
# Slider update behavior
#
# True:
#     redraw while the slider moves
#
# False:
#     redraw after the slider is released
# ------------------------------------------------------------

CONTINUOUS_UPDATE = True

In [3]:
# ============================================================
# CELL 4 — INTERACTIVE TWO-HIDDEN-LAYER NEURAL NETWORK
# ============================================================


# ============================================================
# VALIDATE USER SETTINGS
# ============================================================

if not isinstance(N1, int) or N1 < 1:
    raise ValueError("N1 must be a positive integer.")

if not isinstance(N2, int) or N2 < 1:
    raise ValueError("N2 must be a positive integer.")

if WEIGHT_MAX <= 0:
    raise ValueError("WEIGHT_MAX must be positive.")

if BIAS_MAX <= 0:
    raise ValueError("BIAS_MAX must be positive.")

if SLIDER_STEP <= 0:
    raise ValueError("SLIDER_STEP must be positive.")

if GRID_POINTS < 2:
    raise ValueError("GRID_POINTS must be at least 2.")

if SOFTPLUS_BETA_1 <= 0:
    raise ValueError("SOFTPLUS_BETA_1 must be positive.")

if SOFTPLUS_BETA_2 <= 0:
    raise ValueError("SOFTPLUS_BETA_2 must be positive.")


ACTIVATION_OPTIONS = [
    "tanh",
    "sigmoid",
    "softplus",
    "relu",
    "silu"
]

activation_1_initial = ACTIVATION_LAYER_1.lower()
activation_2_initial = ACTIVATION_LAYER_2.lower()

if activation_1_initial not in ACTIVATION_OPTIONS:
    raise ValueError(
        "ACTIVATION_LAYER_1 must be one of: "
        "tanh, sigmoid, softplus, relu, silu."
    )

if activation_2_initial not in ACTIVATION_OPTIONS:
    raise ValueError(
        "ACTIVATION_LAYER_2 must be one of: "
        "tanh, sigmoid, softplus, relu, silu."
    )


# ============================================================
# ACTIVATION FUNCTION
# ============================================================

def apply_activation(z, activation_name, beta=1.0):
    """
    Apply the selected activation function elementwise.
    """

    activation_name = activation_name.lower()

    if activation_name == "tanh":
        return np.tanh(z)

    if activation_name == "sigmoid":
        z_safe = np.clip(z, -60.0, 60.0)
        return 1.0 / (1.0 + np.exp(-z_safe))

    if activation_name == "softplus":

        if beta <= 0:
            raise ValueError("Softplus beta must be positive.")

        return np.logaddexp(
            0.0,
            beta * z
        ) / beta

    if activation_name == "relu":
        return np.maximum(0.0, z)

    if activation_name == "silu":
        z_safe = np.clip(z, -60.0, 60.0)
        sigmoid_z = 1.0 / (1.0 + np.exp(-z_safe))
        return z * sigmoid_z

    raise ValueError(
        f"Unknown activation function: {activation_name}"
    )


# ============================================================
# CREATE THE DOMAIN [0,1] x [0,1]
# ============================================================

x_values = np.linspace(
    0.0,
    1.0,
    GRID_POINTS
)

y_values = np.linspace(
    0.0,
    1.0,
    GRID_POINTS
)

X, Y = np.meshgrid(
    x_values,
    y_values,
    indexing="xy"
)

# Convert the grid into a list of input points:
#
#     [x1,y1]
#     [x2,y2]
#       ...
#
XY = np.column_stack(
    [
        X.reshape(-1),
        Y.reshape(-1)
    ]
)


# ============================================================
# INITIALIZE THE NEURAL-NETWORK PARAMETERS
# ============================================================

rng = np.random.default_rng(
    RANDOM_SEED
)

initial_weight_limit = min(
    INITIAL_PARAMETER_SCALE,
    WEIGHT_MAX
)

initial_bias_limit = min(
    INITIAL_PARAMETER_SCALE,
    BIAS_MAX
)


def generate_random_state():
    """
    Generate a complete random set of neural-network parameters.
    """

    return {
        # First hidden layer
        #
        # Shape:
        #     W1 = (N1,2)
        #     b1 = (N1,)
        "W1": rng.uniform(
            -initial_weight_limit,
            initial_weight_limit,
            size=(N1, 2)
        ),

        "b1": rng.uniform(
            -initial_bias_limit,
            initial_bias_limit,
            size=N1
        ),

        # Second hidden layer
        #
        # Shape:
        #     W2 = (N2,N1)
        #     b2 = (N2,)
        "W2": rng.uniform(
            -initial_weight_limit,
            initial_weight_limit,
            size=(N2, N1)
        ),

        "b2": rng.uniform(
            -initial_bias_limit,
            initial_bias_limit,
            size=N2
        ),

        # Linear output layer
        #
        # Shape:
        #     W3 = (N2,)
        #     b3 = scalar
        "W3": rng.uniform(
            -initial_weight_limit,
            initial_weight_limit,
            size=N2
        ),

        "b3": float(
            rng.uniform(
                -initial_bias_limit,
                initial_bias_limit
            )
        )
    }


initial_state = generate_random_state()


# ============================================================
# SLIDER SETTINGS
# ============================================================

slider_style = {
    "description_width": "140px"
}

slider_layout = widgets.Layout(
    width="850px"
)


def make_parameter_slider(
    initial_value,
    maximum,
    description
):
    """
    Create one weight or bias slider.
    """

    return widgets.FloatSlider(
        value=float(initial_value),
        min=-float(maximum),
        max=float(maximum),
        step=SLIDER_STEP,
        description=description,
        readout=True,
        readout_format=".2f",
        continuous_update=CONTINUOUS_UPDATE,
        style=slider_style,
        layout=slider_layout
    )


# ============================================================
# CREATE FIRST-LAYER SLIDERS
# ============================================================

W1_sliders = np.empty(
    (N1, 2),
    dtype=object
)

b1_sliders = np.empty(
    N1,
    dtype=object
)

for i in range(N1):

    W1_sliders[i, 0] = make_parameter_slider(
        initial_state["W1"][i, 0],
        WEIGHT_MAX,
        f"W1[{i+1},x]"
    )

    W1_sliders[i, 1] = make_parameter_slider(
        initial_state["W1"][i, 1],
        WEIGHT_MAX,
        f"W1[{i+1},y]"
    )

    b1_sliders[i] = make_parameter_slider(
        initial_state["b1"][i],
        BIAS_MAX,
        f"b1[{i+1}]"
    )


# ============================================================
# CREATE SECOND-LAYER SLIDERS
# ============================================================

W2_sliders = np.empty(
    (N2, N1),
    dtype=object
)

b2_sliders = np.empty(
    N2,
    dtype=object
)

for j in range(N2):

    for i in range(N1):

        W2_sliders[j, i] = make_parameter_slider(
            initial_state["W2"][j, i],
            WEIGHT_MAX,
            f"W2[{j+1},{i+1}]"
        )

    b2_sliders[j] = make_parameter_slider(
        initial_state["b2"][j],
        BIAS_MAX,
        f"b2[{j+1}]"
    )


# ============================================================
# CREATE OUTPUT-LAYER SLIDERS
# ============================================================

W3_sliders = np.empty(
    N2,
    dtype=object
)

for j in range(N2):

    W3_sliders[j] = make_parameter_slider(
        initial_state["W3"][j],
        WEIGHT_MAX,
        f"W3[1,{j+1}]"
    )


b3_slider = make_parameter_slider(
    initial_state["b3"],
    BIAS_MAX,
    "b3"
)


# ============================================================
# ACTIVATION CONTROLS
# ============================================================

activation_1_dropdown = widgets.Dropdown(
    options=ACTIVATION_OPTIONS,
    value=activation_1_initial,
    description="Layer 1 act.",
    style=slider_style,
    layout=widgets.Layout(width="450px")
)

activation_2_dropdown = widgets.Dropdown(
    options=ACTIVATION_OPTIONS,
    value=activation_2_initial,
    description="Layer 2 act.",
    style=slider_style,
    layout=widgets.Layout(width="450px")
)


beta_1_slider = widgets.FloatSlider(
    value=SOFTPLUS_BETA_1,
    min=0.1,
    max=10.0,
    step=0.1,
    description="Softplus beta 1",
    continuous_update=CONTINUOUS_UPDATE,
    style=slider_style,
    layout=widgets.Layout(width="650px")
)

beta_2_slider = widgets.FloatSlider(
    value=SOFTPLUS_BETA_2,
    min=0.1,
    max=10.0,
    step=0.1,
    description="Softplus beta 2",
    continuous_update=CONTINUOUS_UPDATE,
    style=slider_style,
    layout=widgets.Layout(width="650px")
)


# ============================================================
# READ CURRENT SLIDER VALUES
# ============================================================

def read_parameters():
    """
    Read all current weight and bias values from the sliders.
    """

    W1 = np.array(
        [
            [
                W1_sliders[i, 0].value,
                W1_sliders[i, 1].value
            ]
            for i in range(N1)
        ],
        dtype=float
    )

    b1 = np.array(
        [
            b1_sliders[i].value
            for i in range(N1)
        ],
        dtype=float
    )

    W2 = np.array(
        [
            [
                W2_sliders[j, i].value
                for i in range(N1)
            ]
            for j in range(N2)
        ],
        dtype=float
    )

    b2 = np.array(
        [
            b2_sliders[j].value
            for j in range(N2)
        ],
        dtype=float
    )

    W3 = np.array(
        [
            W3_sliders[j].value
            for j in range(N2)
        ],
        dtype=float
    )

    b3 = float(
        b3_slider.value
    )

    return W1, b1, W2, b2, W3, b3


# ============================================================
# NEURAL-NETWORK FORWARD PASS
# ============================================================

def evaluate_network():
    """
    Evaluate the two-hidden-layer neural network.

    Layer 1:
        Z1 = XY W1^T + b1
        A1 = activation_1(Z1)

    Layer 2:
        Z2 = A1 W2^T + b2
        A2 = activation_2(Z2)

    Output:
        U = A2 W3 + b3
    """

    W1, b1, W2, b2, W3, b3 = read_parameters()

    # First hidden layer
    Z1 = XY @ W1.T + b1

    A1 = apply_activation(
        Z1,
        activation_1_dropdown.value,
        beta_1_slider.value
    )

    # Second hidden layer
    Z2 = A1 @ W2.T + b2

    A2 = apply_activation(
        Z2,
        activation_2_dropdown.value,
        beta_2_slider.value
    )

    # Linear output layer
    U = A2 @ W3 + b3

    return U.reshape(
        GRID_POINTS,
        GRID_POINTS
    )


# ============================================================
# PLOT AND STATUS OUTPUT AREAS
# ============================================================

plot_output = widgets.Output(
    layout=widgets.Layout(
        width="100%"
    )
)

status_output = widgets.HTML()

parameters_are_being_set = False


# ============================================================
# REDRAW THE 3D SURFACE
# ============================================================

def redraw_plot(change=None):
    """
    Recalculate the neural-network output and redraw
    the interactive Plotly 3D surface.
    """

    if parameters_are_being_set:
        return

    U = evaluate_network()

    u_min = float(np.min(U))
    u_max = float(np.max(U))

    if np.isclose(u_min, u_max):

        padding = 0.1 * max(
            1.0,
            abs(u_min)
        )

    else:

        padding = 0.08 * (
            u_max - u_min
        )

    z_lower = u_min - padding
    z_upper = u_max + padding

    figure = go.Figure(
        data=[
            go.Surface(
                x=X,
                y=Y,
                z=U,
                colorscale="Viridis",
                cmin=z_lower,
                cmax=z_upper,
                colorbar=dict(
                    title="u(x,y)"
                ),
                hovertemplate=(
                    "x=%{x:.3f}"
                    "<br>y=%{y:.3f}"
                    "<br>u=%{z:.5f}"
                    "<extra></extra>"
                )
            )
        ]
    )

    figure.update_layout(
        title=(
            "Two-hidden-layer neural network: "
            f"{activation_1_dropdown.value} → "
            f"{activation_2_dropdown.value}"
        ),

        width=PLOT_WIDTH,
        height=PLOT_HEIGHT,

        margin=dict(
            l=20,
            r=20,
            t=70,
            b=20
        ),

        scene=dict(
            xaxis=dict(
                title="x",
                range=[0.0, 1.0]
            ),

            yaxis=dict(
                title="y",
                range=[0.0, 1.0]
            ),

            zaxis=dict(
                title="u_theta(x,y)",
                range=[
                    z_lower,
                    z_upper
                ]
            ),

            aspectratio=dict(
                x=1.0,
                y=1.0,
                z=0.8
            ),

            camera=dict(
                eye=dict(
                    x=1.5,
                    y=1.5,
                    z=1.2
                )
            )
        )
    )

    parameter_count = (
        N1 * N2
        + 3 * N1
        + 2 * N2
        + 1
    )

    status_output.value = (
        f"<b>Output range:</b> "
        f"[{u_min:.6g}, {u_max:.6g}]"
        f"&nbsp;&nbsp;&nbsp;"
        f"<b>Adjustable parameters:</b> "
        f"{parameter_count}"
    )

    with plot_output:

        clear_output(wait=True)

        figure.show(
            renderer="colab",
            config={
                "responsive": True,
                "displaylogo": False,
                "scrollZoom": True
            }
        )


# ============================================================
# CONNECT CONTROLS TO THE PLOT
# ============================================================

all_parameter_sliders = (
    list(W1_sliders.reshape(-1))
    + list(b1_sliders)
    + list(W2_sliders.reshape(-1))
    + list(b2_sliders)
    + list(W3_sliders)
    + [b3_slider]
)

all_controls = (
    all_parameter_sliders
    + [
        activation_1_dropdown,
        activation_2_dropdown,
        beta_1_slider,
        beta_2_slider
    ]
)

for control in all_controls:

    control.observe(
        redraw_plot,
        names="value"
    )


# ============================================================
# PARAMETER BUTTONS
# ============================================================

randomize_button = widgets.Button(
    description="Randomize",
    button_style="info"
)

reset_button = widgets.Button(
    description="Reset"
)

zero_button = widgets.Button(
    description="Set all to zero",
    button_style="warning"
)


def set_parameter_state(state):
    """
    Assign a complete parameter state to the sliders.
    """

    global parameters_are_being_set

    parameters_are_being_set = True

    try:

        for i in range(N1):

            W1_sliders[i, 0].value = float(
                state["W1"][i, 0]
            )

            W1_sliders[i, 1].value = float(
                state["W1"][i, 1]
            )

            b1_sliders[i].value = float(
                state["b1"][i]
            )

        for j in range(N2):

            for i in range(N1):

                W2_sliders[j, i].value = float(
                    state["W2"][j, i]
                )

            b2_sliders[j].value = float(
                state["b2"][j]
            )

            W3_sliders[j].value = float(
                state["W3"][j]
            )

        b3_slider.value = float(
            state["b3"]
        )

    finally:

        parameters_are_being_set = False

    redraw_plot()


def randomize_parameters(button):
    set_parameter_state(
        generate_random_state()
    )


def reset_parameters(button):
    set_parameter_state(
        initial_state
    )


def zero_parameters(button):

    zero_state = {
        "W1": np.zeros((N1, 2)),
        "b1": np.zeros(N1),
        "W2": np.zeros((N2, N1)),
        "b2": np.zeros(N2),
        "W3": np.zeros(N2),
        "b3": 0.0
    }

    set_parameter_state(
        zero_state
    )


randomize_button.on_click(
    randomize_parameters
)

reset_button.on_click(
    reset_parameters
)

zero_button.on_click(
    zero_parameters
)


# ============================================================
# BUILD ACTIVATION PANEL
# ============================================================

activation_panel = widgets.VBox(
    [
        widgets.HTML(
            "<h2>Activation functions</h2>"
        ),

        activation_1_dropdown,
        activation_2_dropdown,
        beta_1_slider,
        beta_2_slider
    ],
    layout=widgets.Layout(
        width="100%",
        border="1px solid #cccccc",
        padding="12px",
        margin="5px 0px 15px 0px"
    )
)


# ============================================================
# BUILD LAYER 1 SLIDER PANEL
# ============================================================

layer_1_items = [
    widgets.HTML(
        "<h2>Layer 1 weights and biases</h2>"
        "<p>"
        "W1[i,x] multiplies x, "
        "W1[i,y] multiplies y, "
        "and b1[i] is the bias."
        "</p>"
    )
]

for i in range(N1):

    layer_1_items.append(
        widgets.HTML(
            f"<h3>Layer 1 node {i+1}</h3>"
        )
    )

    layer_1_items.extend(
        [
            W1_sliders[i, 0],
            W1_sliders[i, 1],
            b1_sliders[i]
        ]
    )


layer_1_panel = widgets.VBox(
    layer_1_items,
    layout=widgets.Layout(
        width="100%",
        border="1px solid #cccccc",
        padding="12px",
        margin="5px 0px 15px 0px"
    )
)


# ============================================================
# BUILD LAYER 2 SLIDER PANEL
# ============================================================

layer_2_items = [
    widgets.HTML(
        "<h2>Layer 2 weights and biases</h2>"
        "<p>"
        "W2[j,i] connects Layer 1 node i "
        "to Layer 2 node j."
        "</p>"
    )
]

for j in range(N2):

    layer_2_items.append(
        widgets.HTML(
            f"<h3>Layer 2 node {j+1}</h3>"
        )
    )

    for i in range(N1):

        layer_2_items.append(
            W2_sliders[j, i]
        )

    layer_2_items.append(
        b2_sliders[j]
    )


layer_2_panel = widgets.VBox(
    layer_2_items,
    layout=widgets.Layout(
        width="100%",
        border="1px solid #cccccc",
        padding="12px",
        margin="5px 0px 15px 0px"
    )
)


# ============================================================
# BUILD OUTPUT-LAYER SLIDER PANEL
# ============================================================

output_items = [
    widgets.HTML(
        "<h2>Output-layer weights and bias</h2>"
        "<p>"
        "W3[1,j] connects Layer 2 node j "
        "to the scalar output."
        "</p>"
    )
]

for j in range(N2):

    output_items.append(
        W3_sliders[j]
    )

output_items.append(
    b3_slider
)


output_panel = widgets.VBox(
    output_items,
    layout=widgets.Layout(
        width="100%",
        border="1px solid #cccccc",
        padding="12px",
        margin="5px 0px 15px 0px"
    )
)


# ============================================================
# BUTTON PANEL
# ============================================================

button_panel = widgets.HBox(
    [
        randomize_button,
        reset_button,
        zero_button
    ],
    layout=widgets.Layout(
        width="100%",
        margin="5px 0px 15px 0px"
    )
)


# ============================================================
# DISPLAY ALL CONTROLS AND THE PLOT
# ============================================================

display(
    widgets.HTML(
        "<h1>Neural-Network Manipulate</h1>"
        "<p>"
        "Move any weight or bias slider to redraw the "
        "interactive 3D surface."
        "</p>"
    )
)

display(activation_panel)
display(button_panel)
display(status_output)

# All weight and bias sliders are displayed directly.
display(layer_1_panel)
display(layer_2_panel)
display(output_panel)

display(
    widgets.HTML(
        "<h2>Interactive 3D output</h2>"
    )
)

display(plot_output)

# Draw the initial 3D surface.
redraw_plot()

HTML(value='<h1>Neural-Network Manipulate</h1><p>Move any weight or bias slider to redraw the interactive 3D s…

HTML(value='')

HTML(value='<h2>Interactive 3D output</h2>')

Output(layout=Layout(width='100%'))